# Rozwiązania: Behavior Trees + Nav2

Uruchom wszystkie komórki w kontenerze ROS Jazzy. Nie potrzeba Gazebo ani aktywnych serwerów ROS. Korzystamy z tych samych liści i tej samej struktury co demo w zamku; fałszywy navigator steruje wynikami i zapisuje kolejność efektów.


In [ ]:
from pathlib import Path
import sys

# Działa z katalogu exercises, .solutions oraz katalogu źródeł warsztatu.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd()/"exercises",
              Path.cwd()/"jupyter_notebooks"/"exercises",
              Path("/home/ubuntu/turtlebot3_ws/src/jupyter_notebooks/exercises")]
exercise_dir = next(path for path in candidates if (path/"trees_nav.py").is_file())
if str(exercise_dir) not in sys.path:
    sys.path.insert(0, str(exercise_dir))

import py_trees
import trees_nav
from test_nav_bt import make_fake_demo
from nav2_simple_commander.robot_navigator import TaskResult

Status = py_trees.common.Status
print("Gotowe: nie uruchamiamy Gazebo ani serwerów Nav2.")

from bt_visuals import show_tree


## A1. Przerwanie i własność

Selektor tickuje nową akcję przed unieważnieniem starej. Nowy właściciel anuluje stare zadanie przed wysłaniem celu; stary liść podczas INVALID zwalnia tylko zadanie, którego nadal jest właścicielem.


In [ ]:
demo_fake, nav, bt = make_fake_demo()
bt.tick()
patrol_leaf = demo_fake.navigation_owner
calls_before = list(nav.calls)
demo_fake.request_goal()
assert nav.calls == calls_before
bt.tick()
assert nav.calls == [("send", 1.0, 0.0), ("cancel",), ("send", 9.0, 0.0)]
assert patrol_leaf.status == Status.INVALID and not patrol_leaf.goal_sent
goal_leaf = demo_fake.navigation_owner
calls_before = list(nav.calls)
demo_fake.request_stop()
assert nav.calls == calls_before
bt.tick()
assert nav.calls[-2:] == [("cancel",), ("zero_velocity",)]
assert goal_leaf.status == Status.INVALID and not goal_leaf.goal_sent
assert demo_fake.navigation_owner is None
print(nav.calls)

show_tree(bt.root, "Nav2 bez symulatora: priorytety i recovery")


## A2. Odzyskiwanie zapisane w strukturze

Pamięć selektora utrzymuje drugą próbę. Pamięć sekwencji zapobiega ponownemu czyszczeniu costmap przy każdym ticku trwającej drugiej próby.


In [ ]:
demo_fake, nav, _ = make_fake_demo()
demo_fake.request_goal()
retry = py_trees.composites.Sequence(name="Odzyskiwanie", memory=True)
retry.add_children([trees_nav.ClearCostmaps(demo_fake),
                    trees_nav.NavigateToGoal(demo_fake, "Próba 2")])
recovery = py_trees.composites.Selector(name="Cel z jedną ponowną próbą", memory=True)
recovery.add_children([trees_nav.NavigateToGoal(demo_fake, "Próba 1"), retry])
bt = py_trees.trees.BehaviourTree(recovery)
bt.tick()
nav.complete(TaskResult.FAILED)
bt.tick()
for _ in range(3):
    bt.tick()
assert nav.calls == [("send", 9.0, 0.0), ("clear",), ("send", 9.0, 0.0)]
assert recovery.status == Status.RUNNING
nav.complete(TaskResult.SUCCEEDED)
bt.tick()
assert recovery.status == Status.SUCCESS
assert demo_fake.navigation_owner is None
print(py_trees.display.unicode_tree(recovery, show_status=True))


## A3. Mutacje pamięci

Korzeń z pamięcią omija gałąź STOP: warunek patrolu nadal wykrywa zmianę i anuluje zadanie, ale nie publikuje zerowej prędkości; korzeń zwraca FAILURE. Sekwencja GOAL z pamięcią pomija ponowne sprawdzenie swojego warunku, więc żądanie powrotu do patrolu nie przerywa aktywnego celu.


In [ ]:
for root_memory in (False, True):
    demo_fake, nav, bt = make_fake_demo()
    bt.root.memory = root_memory
    bt.tick()
    demo_fake.request_stop()
    bt.tick()
    assert ("cancel",) in nav.calls
    assert (("zero_velocity",) in nav.calls) == (not root_memory)
    assert bt.root.status == (Status.FAILURE if root_memory else Status.RUNNING)
    print("root memory:", root_memory, nav.calls)
    bt.root.stop(Status.INVALID)

for guard_memory in (False, True):
    demo_fake, nav, bt = make_fake_demo()
    bt.root.children[1].memory = guard_memory
    demo_fake.request_goal()
    bt.tick()
    demo_fake.request_patrol()
    bt.tick()
    expected = ([("send", 9.0, 0.0)] if guard_memory else
                [("send", 9.0, 0.0), ("cancel",), ("send", 1.0, 0.0)])
    assert nav.calls == expected
    print("GOAL guard memory:", guard_memory, nav.calls)
    bt.root.stop(Status.INVALID)


## A4. Sukces, dwie porażki i progres patrolu

Sukces zmienia intencję na PATROL; patrol wchodzi dopiero przy kolejnym ticku. Po dwóch porażkach drzewo przechodzi do STOPPED zamiast ponawiać bez końca. Indeks patrolu aktualizuje osobny liść wyłącznie po sukcesie nawigacji.


In [ ]:
demo_fake, nav, bt = make_fake_demo()
demo_fake.request_goal()
bt.tick()
nav.complete()
bt.tick()
assert demo_fake.state.snapshot()[0] == trees_nav.DemoMode.PATROL
assert nav.calls == [("send", 9.0, 0.0)]
bt.tick()
assert nav.calls[-1] == ("send", 1.0, 0.0)
bt.root.stop(Status.INVALID)

# Dwie nieudane próby: dokładnie jedno czyszczenie i bez dalszych wysłań.
demo_fake, nav, bt = make_fake_demo()
demo_fake.request_goal()
bt.tick()
for _ in range(2):
    nav.complete(TaskResult.FAILED)
    bt.tick()
assert demo_fake.state.snapshot()[0] == trees_nav.DemoMode.STOPPED
for _ in range(5):
    bt.tick()
assert nav.calls == [("send", 9.0, 0.0), ("clear",), ("send", 9.0, 0.0), ("zero_velocity",)]

# Niepowodzenie nie przesuwa indeksu, sukces przesuwa, pełna pętla liczy cykl.
demo_fake, nav, bt = make_fake_demo()
bt.tick()
nav.complete(TaskResult.FAILED)
bt.tick()
assert demo_fake.patrol_waypoint_index == 0
nav.complete()
bt.tick()
assert demo_fake.patrol_waypoint_index == 1
bt.tick()
assert nav.calls[-1] == ("send", 2.0, 0.0)
nav.complete()
bt.tick()
assert demo_fake.patrol_waypoint_index == 0
assert demo_fake.state.snapshot()[2] == 1
print("Wszystkie scenariusze spełniły kontrakt.")


## Pełna regresja

Dodatkowe przypadki obejmują odrzucenie celu przez Nav2, przerwanie powrotu do patrolu, sprzątanie oraz ochronę intencji przed starym wynikiem.


In [ ]:
import unittest
from test_nav_bt import NavigationTreeTests
result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(NavigationTreeTests)
)
assert result.wasSuccessful()
